# 00 - Inventario de fuentes

Auditoría de los 5 archivos de `data/raw/`. Este notebook solo audita:
la limpieza real vive en `src/ingest.py` (`load_base_accidentes`,
`load_historico_accidentes`, `load_accidentes_tablero`, `load_incidentes`)
y `src/clean.py`. Ver el detalle completo en `reports/diccionario_datos.md`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import RAW_DIR

for f in sorted(RAW_DIR.glob("*.xlsx")):
    print(f.name)
    print("  hojas:", pd.ExcelFile(f).sheet_names)

BASE CALCULO INDICADORES 2024.xlsx
  hojas: ['INDICADORES', 'Hoja2', 'ACCIDENTES', 'HH', 'DÍAS SIN', 'Proyección', 'Hoja1', 'Hoja3']
Base Accidentes 2024_Todo Alicorp.xlsx


  hojas: ['Hoja9', 'Base', 'Hoja5', 'Desplegables', 'Tablas', 'Hoja1', 'Tablas (2)', 'Hoja3', 'Hoja2', 'Hoja4', 'Hoja6', 'Hoja7', 'Hoja8']
Base de Accidentes 2023 _ Total.xlsx
  hojas: ['Base', 'Tipo', 'Lesión', 'Hoja2', 'Detalle1', 'Tipo Trabajador', 'Gráficos', 'Hoja1', 'Hoja4', 'Causas Inmediatas', 'Causas Básicas1', 'Causas Básicas2', 'Resumen']
Resultados SST 2023 v06final.xlsx


  hojas: ['PROGRAMA 2010', 'ENE', 'FEB', 'MAR', 'NVO IOM', 'HISTORICO ACCIDENTES', 'HH 3EROS', 'HH ALICORP', 'HORAS HOMBRE', 'ACCIDENTES INCAPACITANTES', 'INCIDENTE MATERIAL', 'DIAS PERDIDOS', 'BSC CORP', 'RESUMEN IOM ALICORP', 'IND EXTRANJEROS', 'IOM', 'MTBA', 'Graficos', 'Otros indicadores', 'Graphics', 'INDICADORES', 'horas alicorp ', 'Graficos 2012-2016', 'Graf FP - Activ', 'Dias Sin Accidentes', 'estadistica dias sin accidente', 'Graf Det', 'COPSA', 'INDECI', 'CURSOS PERM TRAB', 'mapeo procesos', 'POR ACTIVIDAD', 'EVALUACIONES - REASEG', 'CAPACITACION OBR', 'JABONERIA', 'ERGONOMIA', 'SEGUIMIENTO IPER', 'ESTADISTICA A JUlIO', 'Hoja3', 'PROMOTORES', 'IPER', 'VOLUMENES PRODUCCION', 'BCP']
Tablero de Accidentes e incidentes.xlsx
  hojas: ['1° Incidentes ', '2° Accidentes', 'Hoja2', 'VISUAL', 'Hoja1', 'Hoja3', '2024', '2023', 'TABLERO']


## Hallazgo: 4 de los 5 archivos tienen (o contienen una hoja con) registros fila-por-accidente/incidente

- `Base de Accidentes 2023 _ Total.xlsx` (hoja `Base`) — accidentes 2023
- `Base Accidentes 2024_Todo Alicorp.xlsx` (hoja `Base`) — accidentes 2024
- `Resultados SST 2023 v06final.xlsx` (hoja `HISTORICO ACCIDENTES`) — accidentes 2012-2022
- `Tablero de Accidentes e incidentes.xlsx` (hoja `2° Accidentes`) — accidentes 2025-2026
- `Tablero de Accidentes e incidentes.xlsx` (hoja `1° Incidentes`) — incidentes 2025-2026 (evento distinto a un accidente)

Solo `BASE CALCULO INDICADORES 2024.xlsx` es puramente un archivo de
indicadores agregados, sin registros fila-por-evento — no se procesó. Las
demás ~50 hojas de `Resultados SST 2023 v06final.xlsx` y las hojas
`TABLERO`/`VISUAL` del tablero son vistas/resúmenes, tampoco se procesaron.
Ver la tabla de fuentes completa en `reports/diccionario_datos.md`.

In [2]:
from src.ingest import load_raw

df23_raw = load_raw("Base de Accidentes 2023 _ Total.xlsx", sheet_name="Base")
df24_raw = load_raw("Base Accidentes 2024_Todo Alicorp.xlsx", sheet_name="Base")

print("2023 crudo:", df23_raw.shape, "-> filas con Fecha real:", df23_raw["Fecha"].notna().sum())
print("2024 crudo:", df24_raw.shape)

2023 crudo: (306, 41) -> filas con Fecha real: 107
2024 crudo: (281, 57)


/Users/pierre/Desktop/Applied-Machine-Learning/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/pierre/Desktop/Applied-Machine-Learning/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Hallazgos de calidad de datos (documentados en `reports/diccionario_datos.md`)

1. **2023**: de 306 filas, solo 107 son datos reales. Las 199 restantes son
   residuo de formato de Excel (solo la columna `Sem` traía valor).
2. **2024**: 11 columnas `Unnamed: 46..56` vacías (artefacto de formato),
   descartadas.
3. **Fechas mixtas**: algunas celdas de `Fecha` no tenían formato de fecha
   en el Excel original y llegaban como número de serie de Excel en vez de
   `datetime` — `src/ingest.py` lo corrige (`_parse_mixed_excel_date`).
4. **Nombres/apellidos**: se reemplazan por `id_persona` (hash) antes de
   guardar nada — nunca se escribe un nombre real fuera de `data/raw/`.
5. **HISTORICO 2012-2022**: 85 de 1028 filas tienen `Fecha` mal escrita en
   el Excel original (mes/año inválido) y quedan como `NaT` — no se adivinó
   ninguna fecha.
6. **"2° Accidentes" (2025-2026)**: 14 columnas de seguimiento con códigos
   mixtos e inconsistentes (`T1`/`T2`, `SI`/`NO` no uniforme) se descartaron
   por no poder interpretarlas con confianza.
7. **"1° Incidentes" (2025-2026)**: no trae nombre/DNI de ninguna persona,
   no requirió anonimización.

In [3]:
from src.clean import report_nulls
from src.ingest import load_base_accidentes, save_processed

df23 = load_base_accidentes("2023")
df24 = load_base_accidentes("2024")

print("2023 limpio:", df23.shape)
print("2024 limpio:", df24.shape)

save_processed(df23, "accidentes_2023")
save_processed(df24, "accidentes_2024")

/Users/pierre/Desktop/Applied-Machine-Learning/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


2023 limpio: (107, 40)
2024 limpio: (281, 45)


/Users/pierre/Desktop/Applied-Machine-Learning/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


PosixPath('/Users/pierre/Desktop/Applied-Machine-Learning/data/processed/accidentes_2024.csv')

In [4]:
# Nulos por columna (NO se imputa nada aqui, solo se reporta).
# Ver decision de cada columna en reports/diccionario_datos.md.
print("--- 2023 ---")
display(report_nulls(df23))
print("--- 2024 ---")
display(report_nulls(df24))

--- 2023 ---


,n_nulos,pct_nulos
item,100,93.46
nro_rom,98,91.59
fecha_ingreso_labores,96,89.72
tiempo_experiencia_meses,94,87.85
parte_cuerpo_afectada,90,84.11
actividad_realizada,90,84.11
dia,90,84.11
mes,90,84.11
anio,90,84.11
hora,90,84.11


--- 2024 ---


,n_nulos,pct_nulos
es_considerado,193,68.68
fecha_ingreso_labores,120,42.70
tiempo_experiencia_meses,100,35.59
antiguedad,92,32.74
edad_anios,79,28.11
nro_rom,58,20.64
fuente_peligro,46,16.37
tipo_contrato,39,13.88
cargo_ansi,26,9.25
diagnostico_medico,26,9.25


## Pendiente

- [ ] `BASE CALCULO INDICADORES 2024.xlsx` sigue sin procesar (son
      indicadores ya agregados, no filas por evento) — evaluar si sirve
      solo como referencia cruzada para validar los totales del EDA.
- [ ] Decidir con el equipo qué hacer con las columnas que superan 80% de
      nulos en cualquiera de los 5 CSV (ver `reports/diccionario_datos.md`).
- [ ] Antes de entrenar un modelo, decidir cómo unificar los esquemas de las
      5 fuentes (columnas comunes vs. entrenar por separado) — no son
      idénticos entre sí.
- [ ] Revisar `descripcion_accidente`, `descripcion_incidente` y
      `danos_reales_o_potenciales` antes de compartirlas (pueden mencionar
      nombres en el texto libre — no se anonimizaron automáticamente).

In [5]:
from src.ingest import load_accidentes_tablero, load_historico_accidentes, load_incidentes

df_historico = load_historico_accidentes()
df_2025_2026 = load_accidentes_tablero()
df_incidentes = load_incidentes()

print("HISTORICO 2012-2022:", df_historico.shape)
print("Accidentes 2025-2026:", df_2025_2026.shape)
print("Incidentes 2025-2026:", df_incidentes.shape)

save_processed(df_historico, "accidentes_historico_2012_2022")
save_processed(df_2025_2026, "accidentes_2025_2026")
save_processed(df_incidentes, "incidentes_2025_2026")

HISTORICO 2012-2022: (1028, 27)
Accidentes 2025-2026: (151, 29)
Incidentes 2025-2026: (340, 9)


PosixPath('/Users/pierre/Desktop/Applied-Machine-Learning/data/processed/incidentes_2025_2026.csv')

In [6]:
# Nulos por columna (NO se imputa nada aqui, solo se reporta).
# Ver decision de cada columna en reports/diccionario_datos.md.
print("--- HISTORICO 2012-2022 ---")
display(report_nulls(df_historico))
print("--- Accidentes 2025-2026 ---")
display(report_nulls(df_2025_2026))
print("--- Incidentes 2025-2026 ---")
display(report_nulls(df_incidentes))

--- HISTORICO 2012-2022 ---


,n_nulos,pct_nulos
tipo_contacto,1028,100.00
nota_midot,1014,98.64
cargo_ansi,1004,97.67
modalidad,984,95.72
nro_rom,925,89.98
contrata,753,73.25
area_responsabilidad,501,48.74
edad_anios,381,37.06
sexo,351,34.14
experiencia_puesto,334,32.49


--- Accidentes 2025-2026 ---


,n_nulos,pct_nulos
horas_trabajadas_turno,149,98.68
es_hard_stop,120,79.47
edad_anios,117,77.48
tiempo_empresa_anios,108,71.52
tiempo_empresa_meses,107,70.86
area,101,66.89
nro_rom,84,55.63
hora,83,54.97
causa_basica,74,49.01
turno,24,15.89


--- Incidentes 2025-2026 ---


,n_nulos,pct_nulos
es_verificado,328,96.47
danos_reales_o_potenciales,247,72.65
fuente_peligro,247,72.65
empresa,2,0.59
descripcion_incidente,1,0.29


## Pendiente

- [ ] `BASE CALCULO INDICADORES 2024.xlsx` sigue sin procesar (son
      indicadores ya agregados, no filas por evento) — es el único de los 5
      archivos de `data/raw/` que queda sin auditar/procesar; los otros 3
      (HISTORICO ACCIDENTES, 2° Accidentes, 1° Incidentes) ya se cargan
      arriba.
- [ ] Decidir con el equipo qué hacer con las columnas de `accidentes_2023`
      que superan 80% de nulos (ver `reports/diccionario_datos.md`).
- [ ] Revisar `descripcion_accidente` antes de compartirla (puede mencionar
      nombres en el texto libre — no se anonimizó automáticamente).